# Pipeline

DB -> DBT -> marts(cubes) -> metricflow -> LLM agents

https://github.com/datamindedbe/blog-tpcds-dbt-duckdb/tree/main

```
uv sync
```

In [ ]:
# # run this to generate index for values in the hierarchy yaml files

# import duckdb
# from src.hierarchy_duckdb import build_tree_with_stats
# from pathlib import Path
# proj_path = Path().resolve()
# data_path = proj_path / 'data'
# duckdb_conn = duckdb.connect(database=str(proj_path / 'tpcds/tpcds.db'))
# index_path = data_path / 'index'
# for yaml_path in (data_path / 'hierarchy').glob('*.yaml'):
#     tree = build_tree_with_stats(yaml_path, index_path, duckdb_conn)
#     with (data_path / 'hierarchy' / f"{yaml_path.stem}.json").open('w') as f:
#         f.write(tree.to_json())

In [ ]:
# # run this only once to generate the TPC-DS data
import duckdb

con = duckdb.connect(database='./tpcds/tpcds.db')
con.execute('INSTALL tpcds;')
con.execute('LOAD tpcds;')
# con.execute("CALL dsdgen(sf = 1);")  # run only once generate data with scale factor 1 (1GB)

In [2]:
from pathlib import Path
from src.schema_processor import display_graph

data_path = Path('./data')

In [3]:
display_graph(data_path)

Output()

In [4]:
from src.schema_processor import SchemaExplorer

explorer = SchemaExplorer(data_path)
attr_results = explorer.search_attribute('star', 'store_sales', 'd_fy_year')  # d_fy_year, s_store_id
attr_results

[{'dimension': 'date_dim',
  'attribute': 'd_fy_year',
  'path': [{'type': 'dimension', 'name': 'date_dim', 'label': 'date_dim'},
   {'type': 'level', 'name': 'd_date', 'label': 'Fiscal'},
   {'type': 'level', 'name': 'd_date', 'label': 'Date'},
   {'type': 'level', 'name': 'd_fy_week_seq', 'label': 'FY Week Seq'},
   {'type': 'level', 'name': 'd_fy_quarter_seq', 'label': 'FY Quarter Seq'},
   {'type': 'attribute', 'name': 'd_fy_year', 'label': 'FY Year'}],
  'stats': {'count': 73049,
   'null_count': 0,
   'distinct_count': 201,
   'min': 1900,
   'max': 2100,
   'range': [1900, 2100],
   'dtype': 'integer',
   'unique_values': {'type': 'bplustree',
    'values': '/home/jsjang/code/Agent4OLAP/data/index/date_dim__d_fy_year'}}}]

In [5]:
import random
from src.unique_index import UniqueIndex
path = './data/index/date_dim__d_fy_year'
idx = UniqueIndex(path, fast=False)

x = random.sample(list(iter(idx)), k=1)[0]
print("Search for:", x)
o = explorer.search_value('star', 'store_sales', 'd_fy_year', x)
print("Found:", o[0]['found'])
o

Search for: 2037
Found: True


[{'match': {'dimension': 'date_dim',
   'attribute': 'd_fy_year',
   'path': [{'type': 'dimension', 'name': 'date_dim', 'label': 'date_dim'},
    {'type': 'level', 'name': 'd_date', 'label': 'Fiscal'},
    {'type': 'level', 'name': 'd_date', 'label': 'Date'},
    {'type': 'level', 'name': 'd_fy_week_seq', 'label': 'FY Week Seq'},
    {'type': 'level', 'name': 'd_fy_quarter_seq', 'label': 'FY Quarter Seq'},
    {'type': 'attribute', 'name': 'd_fy_year', 'label': 'FY Year'}],
   'stats': {'count': 73049,
    'null_count': 0,
    'distinct_count': 201,
    'min': 1900,
    'max': 2100,
    'range': [1900, 2100],
    'dtype': 'integer',
    'unique_values': {'type': 'bplustree',
     'values': '/home/jsjang/code/Agent4OLAP/data/index/date_dim__d_fy_year'}}},
  'value': '2037',
  'found': True}]